In [4]:
from xgboost import XGBRegressor
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score,mean_absolute_error,mean_squared_error,root_mean_squared_error
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OrdinalEncoder,OneHotEncoder

In [5]:
df=pd.read_csv("D:\ML\datasets\CAR DETAILS FROM CAR DEKHO.csv")

In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4340 entries, 0 to 4339
Data columns (total 8 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   name           4340 non-null   object
 1   year           4340 non-null   int64 
 2   selling_price  4340 non-null   int64 
 3   km_driven      4340 non-null   int64 
 4   fuel           4340 non-null   object
 5   seller_type    4340 non-null   object
 6   transmission   4340 non-null   object
 7   owner          4340 non-null   object
dtypes: int64(3), object(5)
memory usage: 271.4+ KB


In [8]:
cat_col=df.select_dtypes(include='object').columns
df[cat_col]=df[cat_col].astype('category')

In [9]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4340 entries, 0 to 4339
Data columns (total 8 columns):
 #   Column         Non-Null Count  Dtype   
---  ------         --------------  -----   
 0   name           4340 non-null   category
 1   year           4340 non-null   int64   
 2   selling_price  4340 non-null   int64   
 3   km_driven      4340 non-null   int64   
 4   fuel           4340 non-null   category
 5   seller_type    4340 non-null   category
 6   transmission   4340 non-null   category
 7   owner          4340 non-null   category
dtypes: category(5), int64(3)
memory usage: 171.9 KB


In [10]:
df.head()

,name,year,selling_price,km_driven,fuel,seller_type,transmission,owner
0,Maruti 800 AC,2007,60000,70000,Petrol,Individual,Manual,First Owner
1,Maruti Wagon R LXI Minor,2007,135000,50000,Petrol,Individual,Manual,First Owner
2,Hyundai Verna 1.6 SX,2012,600000,100000,Diesel,Individual,Manual,First Owner
3,Datsun RediGO T Option,2017,250000,46000,Petrol,Individual,Manual,First Owner
4,Honda Amaze VX i-DTEC,2014,450000,141000,Diesel,Individual,Manual,Second Owner


In [11]:
df['brand']=df['name'].str.split().str[0]

In [12]:
df.head()

,name,year,selling_price,km_driven,fuel,seller_type,transmission,owner,brand
0,Maruti 800 AC,2007,60000,70000,Petrol,Individual,Manual,First Owner,Maruti
1,Maruti Wagon R LXI Minor,2007,135000,50000,Petrol,Individual,Manual,First Owner,Maruti
2,Hyundai Verna 1.6 SX,2012,600000,100000,Diesel,Individual,Manual,First Owner,Hyundai
3,Datsun RediGO T Option,2017,250000,46000,Petrol,Individual,Manual,First Owner,Datsun
4,Honda Amaze VX i-DTEC,2014,450000,141000,Diesel,Individual,Manual,Second Owner,Honda


In [14]:
X=df.drop(columns=['name','selling_price'])
Y=df['selling_price']

In [15]:
X.head()

,year,km_driven,fuel,seller_type,transmission,owner,brand
0,2007,70000,Petrol,Individual,Manual,First Owner,Maruti
1,2007,50000,Petrol,Individual,Manual,First Owner,Maruti
2,2012,100000,Diesel,Individual,Manual,First Owner,Hyundai
3,2017,46000,Petrol,Individual,Manual,First Owner,Datsun
4,2014,141000,Diesel,Individual,Manual,Second Owner,Honda


In [16]:
xtrain, xtest, ytrain, ytest = train_test_split(
    X,
    Y,
    train_size=0.8,
    random_state=103
)

In [17]:
one_hot = ['fuel', 'seller_type', 'transmission', 'brand']
ordinal = ['owner']

In [18]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            'one_hot',
            OneHotEncoder(handle_unknown='ignore'),
            one_hot
        ),
        (
            'ordinal',
            OrdinalEncoder(categories=[[
                'First Owner',
                'Second Owner',
                'Third Owner',
                'Fourth & Above Owner',
                'Test Drive Car'
            ]]),
            ordinal
        )
    ],
    remainder='passthrough'
)

In [55]:
model = Pipeline([
    ('preprocessor', preprocessor),
    ('model', XGBRegressor(
        n_estimators=110,
        learning_rate=0.1,
        max_depth=2,
        random_state=42
    ))
])

In [56]:
model.fit(xtrain, ytrain)

,steps,"[('preprocessor', ...), ('model', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('one_hot', ...), ('ordinal', ...)]"
,remainder,'passthrough'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [57]:
train_pred = model.predict(xtrain)
test_pred = model.predict(xtest)

In [59]:
train_r2 = r2_score(ytrain, train_pred)
test_r2 = r2_score(ytest, test_pred)

mae = mean_absolute_error(ytest, test_pred)
mse = mean_squared_error(ytest, test_pred)
rmse = root_mean_squared_error(ytest, test_pred)

print("Training R² :", train_r2)
print("Testing R²  :", test_r2)
print("Difference  :", train_r2 - test_r2)

print("\nMAE  :", mae)
print("MSE  :", mse)
print("RMSE :", rmse)

Training R² : 0.8062705993652344
Testing R²  : 0.777700662612915
Difference  : 0.028569936752319336

MAE  : 158723.390625
MSE  : 79455690752.0
RMSE : 281878.84375
